In [ ]:
## Prerequisite Python Modules
#First, some software needs to be loaded into the Python environment.
#link: https://colab.research.google.com/github/AviatorMoser/keras-mnist-tutorial/blob/master/MNIST%20in%20Keras.ipynb#scrollTo=axuLcCRb4w7Y
import numpy as np                   # advanced math library
import matplotlib.pyplot as plt      # MATLAB like plotting routines
import random                        # for generating random numbers

from keras.datasets import mnist     # MNIST dataset is included in Keras
from keras.models import Sequential  # Model type to be used

from keras.layers.core import Dense, Dropout, Activation # Types of layers to be used in our model
from keras.utils import np_utils                         # NumPy related tools


Using TensorFlow backend.


In [ ]:
#Loading Training Data
#The MNIST dataset is conveniently bundled within Keras, and we can easily analyze some of its features in Python.

# The MNIST data is split between 60,000 28 x 28 pixel training images and 10,000 28 x 28 pixel images
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train shape", X_train.shape)
print("y_train shape", y_train.shape)
print("X_test shape", X_test.shape)
print("y_test shape", y_test.shape)

X_train shape (60000, 28, 28)
y_train shape (60000,)
X_test shape (10000, 28, 28)
y_test shape (10000,)


In [ ]:
#Formatting the input data layer
#Instead of a 28 x 28 matrix, we build our network to accept a 784-length vector.

#Each image needs to be then reshaped (or flattened) into a vector.
#We'll also normalize the inputs to be in the range [0-1] rather than [0-255].
#Normalizing inputs is generally recommended, so that any additional dimensions (for other network architectures)
#are of the same scale.

X_train = X_train.reshape(60000, 784) # reshape 60,000 28 x 28 matrices into 60,000 784-length vectors.
X_test = X_test.reshape(10000, 784)   # reshape 10,000 28 x 28 matrices into 10,000 784-length vectors.

X_train = X_train.astype('float32')   # change integers to 32-bit floating point numbers
X_test = X_test.astype('float32')

X_train /= 255                        # normalize each value for each pixel for the entire vector for each input
X_test /= 255

print("Training matrix shape", X_train.shape)
print("Testing matrix shape", X_test.shape)

nb_classes = 10 # number of unique digits

Y_train = np_utils.to_categorical(y_train, nb_classes)
Y_test = np_utils.to_categorical(y_test, nb_classes)

print (Y_train)
print (Y_test)

#np_utils.to_categorical: Converts a class vector (integers) to binary class matrix.
#y = [0, 1, 2, 3]
#print (np_utils.to_categorical(y, num_classes=4))
#

Training matrix shape (60000, 784)
Testing matrix shape (10000, 784)
[[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]]
[[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
# The Sequential model is a linear stack of layers and is very common.

model = Sequential()
# The first hidden layer is a set of 512 nodes (artificial neurons).
# Each node will receive an element from each input vector and apply some weight and bias to it.

#same things as:
#tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
#X_train = X_train.reshape(60000, 784)

model.add(Dense(512, input_shape=(784,))) #(784,) is not a typo -- that represents a 784 length vector!
# An "activation" is a non-linear function applied to the output of the layer above.
# It checks the new value of the node, and decides whether that artifical neuron has fired.
# The Rectified Linear Unit (ReLU) converts all negative inputs to nodes in the next layer to be zero.
# Those inputs are then not considered to be fired.
# Positive values of a node are unchanged.

model.add(Activation('relu'))
# Dropout zeroes a selection of random outputs (i.e., disables their activation)
# Dropout helps protect the model from memorizing or "overfitting" the training data.
model.add(Dropout(0.2))
# The second hidden layer appears identical to our first layer.
# However, instead of each of the 512-node receiving 784-inputs from the input image data,
# they receive 512 inputs from the output of the first 512-node layer.

model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.2))

# The final layer of 10 neurons in fully-connected to the previous 512-node layer.
# The final layer of a FCN should be equal to the number of desired classes (10 in this case).
model.add(Dense(10))

# The "softmax" activation represents a probability distribution over K different possible outcomes.
# Its values are all non-negative and sum to 1.

model.add(Activation('softmax'))

# Summarize the built model

model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_1 (Dense)              (None, 512)               401920    
_________________________________________________________________
activation_1 (Activation)    (None, 512)               0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 512)               0         
_________________________________________________________________
dense_2 (Dense)              (None, 512)               262656    
_________________________________________________________________
activation_2 (Activation)    (None, 512)               0         
_________________________________________________________________
dropout_2 (Dropout)          (None, 512)               0         
_________________________________________________________________
dense_3 (Dense)              (None, 10)               

In [ ]:
# Let's use the Adam optimizer for learning
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
#Train the model!
model.fit(X_train, Y_train,
          batch_size=128, epochs=5,
          verbose=1)

Epoch 1/5
60000/60000 [==============================] - 9s 149us/step - loss: 0.2601 - accuracy: 0.9249
Epoch 2/5
60000/60000 [==============================] - 9s 153us/step - loss: 0.2464 - accuracy: 0.9281
Epoch 3/5
60000/60000 [==============================] - 9s 155us/step - loss: 0.2362 - accuracy: 0.9314
Epoch 4/5
60000/60000 [==============================] - 9s 157us/step - loss: 0.2262 - accuracy: 0.9342
Epoch 5/5
60000/60000 [==============================] - 10s 160us/step - loss: 0.2182 - accuracy: 0.9360


In [ ]:
#Evaluate Model's Accuracy on Test Data
score = model.evaluate(X_test, Y_test, batch_size=128)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

10000/10000 [==============================] - 1s 63us/step
Test loss: 0.18077966903448106
Test accuracy: 0.9465000033378601


In [ ]:
#Inspecting the output
# The predict_classes function outputs the highest probability class
# according to the trained classifier for each input example.

predicted_classes = model.predict_classes(X_test)
#predicted_probs = model.predict_proba(X_test)
#print (predicted_probs)

# Check which items we got right / wrong
correct_indices = np.nonzero(predicted_classes == y_test)[0]
incorrect_indices = np.nonzero(predicted_classes != y_test)[0]
print (correct_indices[:9])

for i, correct in enumerate(correct_indices[:9]):
    print("Predicted {}, Class {}".format(predicted_classes[correct], y_test[correct]))

print (incorrect_indices[:9])
print ("#Incorrect prediction")
for i, incorrect in enumerate(incorrect_indices[:9]):
    print("Predicted {}, Class {}".format(predicted_classes[incorrect], y_test[incorrect]))

[0 1 2 3 4 5 6 7 9]
Predicted 7, Class 7
Predicted 2, Class 2
Predicted 1, Class 1
Predicted 0, Class 0
Predicted 4, Class 4
Predicted 1, Class 1
Predicted 4, Class 4
Predicted 9, Class 9
Predicted 9, Class 9
[  8  33 124 149 193 233 241 245 247]
#Incorrect prediction
Predicted 6, Class 5
Predicted 6, Class 4
Predicted 4, Class 7
Predicted 9, Class 2
Predicted 4, Class 9
Predicted 7, Class 8
Predicted 8, Class 9
Predicted 5, Class 3
Predicted 2, Class 4
